In [0]:
# ─── CÉLULA 1 — CONFIGURAÇÃO E REGISTRO DA VIEW ──────────────────────────────────
# Lê a camada Silver e registra como view temporária para uso com Spark SQL puro.
# createOrReplaceTempView() cria uma abstração local da sessão Spark — não persiste no catálogo e é mais eficiente que criar tabelas temporárias no storage.
# Todas as análises a seguir são escritas em SQL para demonstrar fluência na API declarativa do Spark, mais próxima de como analistas de dados consomem os dados.

df = spark.read.table("portfolio.default.telco_silver")
df.createOrReplaceTempView("telco")

print("✔ View 'telco' registrada — pronta para Spark SQL")

✔ View 'telco' registrada — pronta para Spark SQL


In [0]:
# ─── CÉLULA 2 — ANÁLISE 1: CHURN POR TIPO DE CONTRATO ────────────────────────────
# Hipótese central: clientes em contratos mensais cancelam mais que os anuais.
# Contratos month-to-month têm menor custo de saída (sem fidelidade), o que naturalmente eleva a taxa de churn. Esta análise confirma a hipótese e quantifica o impacto — resultado que motiva a inclusão de `Contract` como feature no modelo.

display(spark.sql("""
    SELECT
        Contract,
        COUNT(*)                                    AS total_clientes,
        SUM(Churn)                                  AS total_churn,
        ROUND(SUM(Churn) / COUNT(*) * 100, 1)       AS taxa_churn_pct
    FROM telco
    GROUP BY Contract
    ORDER BY taxa_churn_pct DESC
"""))

Contract,total_clientes,total_churn,taxa_churn_pct
Month-to-month,3875,1655,42.7
One year,1473,166,11.3
Two year,1695,48,2.8


In [0]:
# ─── CÉLULA 3 — ANÁLISE 2: CHURN POR MÉTODO DE PAGAMENTO ────────────────────────
# Pagamento via cheque eletrônico tende a indicar menor engajamento com a empresa — clientes que optam por métodos manuais têm menos "lock-in" psicológico.
# O ticket médio por método expõe se o churn é concentrado em clientes de alto valor.

display(spark.sql("""
    SELECT
        PaymentMethod,
        COUNT(*)                                    AS total_clientes,
        ROUND(SUM(Churn) / COUNT(*) * 100, 1)       AS taxa_churn_pct,
        ROUND(AVG(MonthlyCharges), 2)               AS ticket_medio
    FROM telco
    GROUP BY PaymentMethod
    ORDER BY taxa_churn_pct DESC
"""))

PaymentMethod,total_clientes,taxa_churn_pct,ticket_medio
Electronic check,2365,45.3,76.26
Mailed check,1612,19.1,43.92
Bank transfer (automatic),1544,16.7,67.19
Credit card (automatic),1522,15.2,66.51


In [0]:
# ─── CÉLULA 4 — ANÁLISE 3: PERFIL FINANCEIRO — CHURN VS RETENÇÃO ─────────────────
# Compara o perfil de tenure e cobrança entre clientes que cancelaram e os que ficaram.
# A mediana de tenure (percentil 0.5) é mais robusta que a média em distribuições assimétricas — clientes novos puxam a média para baixo.
# Resultado esperado: clientes que cancelam têm menor tenure e maior cobrança mensal.

display(spark.sql("""
    SELECT
        Churn,
        ROUND(AVG(tenure), 1)                       AS tenure_medio,
        ROUND(AVG(MonthlyCharges), 2)               AS cobranca_mensal_media,
        ROUND(AVG(TotalCharges), 2)                 AS total_pago_medio,
        ROUND(PERCENTILE(tenure, 0.5), 1)           AS tenure_mediano
    FROM telco
    GROUP BY Churn
    ORDER BY Churn
"""))

Churn,tenure_medio,cobranca_mensal_media,total_pago_medio,tenure_mediano
0,37.6,61.27,2549.91,38.0
1,18.0,74.44,1531.8,10.0


In [0]:
# ─── CÉLULA 5 — ANÁLISE 4: NÚMERO DE SERVIÇOS × CHURN ────────────────────────────
# Antecipa a feature `num_services` criada no notebook 04.
# A hipótese é que clientes com mais serviços contratados têm maior custo de saída (precisam reconfigurar múltiplos serviços) e maior satisfação percebida, resultando em menor taxa de churn. Esta análise valida a criação da feature.

display(spark.sql("""
    SELECT
        (CASE WHEN PhoneService    = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN MultipleLines   = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN OnlineSecurity  = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN OnlineBackup    = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN DeviceProtection= 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN TechSupport     = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN StreamingTV     = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN StreamingMovies = 'Yes' THEN 1 ELSE 0 END)  AS num_services,
        COUNT(*)                                                AS total_clientes,
        ROUND(SUM(Churn) / COUNT(*) * 100, 1)                  AS taxa_churn_pct
    FROM telco
    GROUP BY num_services
    ORDER BY num_services
"""))

num_services,total_clientes,taxa_churn_pct
0,80,43.8
1,1701,21.1
2,1188,32.8
3,965,36.5
4,922,31.3
5,908,25.6
6,676,22.5
7,395,12.4
8,208,5.3


In [0]:
# ─── CÉLULA 6 — ANÁLISE 5: TIPO DE INTERNET × CHURN ──────────────────────────────
# Fiber Optic apresenta historicamente alto churn em datasets de telecom, possivelmente por ser um serviço premium com alta concorrência e expectativas elevadas de qualidade. Esta análise confirma o padrão e justifica manter `InternetService` como feature no modelo.

display(spark.sql("""
    SELECT
        InternetService,
        COUNT(*)                                    AS total_clientes,
        ROUND(SUM(Churn) / COUNT(*) * 100, 1)       AS taxa_churn_pct,
        ROUND(AVG(MonthlyCharges), 2)               AS ticket_medio
    FROM telco
    GROUP BY InternetService
    ORDER BY taxa_churn_pct DESC
"""))

InternetService,total_clientes,taxa_churn_pct,ticket_medio
Fiber optic,3096,41.9,91.5
DSL,2421,19.0,58.1
No,1526,7.4,21.08


In [0]:
# ─── CÉLULA 7 — ANÁLISE 6: FAIXAS DE TENURE × VULNERABILIDADE AO CHURN ──────────
# Segmenta clientes por tempo de contrato para identificar o período crítico de churn.
# A feature `is_new_customer` (tenure ≤ 12) criada no notebook 04 é diretamente motivada por esta análise. Clientes nos primeiros 12 meses são os mais vulneráveis e representam a população-alvo de ações de retenção proativas.

display(spark.sql("""
    SELECT
        CASE
            WHEN tenure BETWEEN 0  AND 12 THEN '0-12 meses (novo)'
            WHEN tenure BETWEEN 13 AND 24 THEN '13-24 meses'
            WHEN tenure BETWEEN 25 AND 48 THEN '25-48 meses'
            ELSE '49+ meses (fiel)'
        END                                         AS faixa_tenure,
        COUNT(*)                                    AS total_clientes,
        ROUND(SUM(Churn) / COUNT(*) * 100, 1)       AS taxa_churn_pct
    FROM telco
    GROUP BY faixa_tenure
    ORDER BY taxa_churn_pct DESC
"""))

faixa_tenure,total_clientes,taxa_churn_pct
0-12 meses (novo),2186,47.4
13-24 meses,1024,28.7
25-48 meses,1594,20.4
49+ meses (fiel),2239,9.5


## 📊 Insights do EDA — Principais Fatores de Churn

| Fator               | Observação                                        | Impacto |
|---------------------|---------------------------------------------------|---------|
| Tipo de contrato    | Month-to-month → 42.7% churn vs 2.8% two-year   | 🔴 Alto |
| Tempo de contrato   | Primeiros 12 meses são críticos (47.7% churn)    | 🔴 Alto |
| Internet Fiber      | 41.9% churn — pode indicar problema de qualidade | 🔴 Alto |
| Pagamento eletrônico| 45.3% churn — perfil de baixo engajamento        | 🔴 Alto |
| Nº de serviços      | Mais serviços = maior retenção                    | 🟡 Médio|
| Cobrança mensal     | Quem cancela paga mais (R$74 vs R$61)            | 🟡 Médio|

### Features mais promissoras para o modelo (notebook 04):
- `tenure` e faixas de tenure
- `Contract`
- `InternetService`
- `PaymentMethod`
- `num_services` (feature nova a criar)
- `MonthlyCharges`